# Classification of Dementia type from FDG-PET images: old-school strategy

In this notebook we will show how to classify PET images in dementia or healthy, and the subtype of dementia using a very simple strategy:
estimating the spatial pattern and computing its expression

These PET images are saved in the nifti file format. Start by learning how to read them using nibabel

In [ ]:
import nibabel as nib
import numpy as np
import glob
import os
import matplotlib.pyplot as plt

# Task 1: read one Nifti file

you will real the file using nib.load, which returns a structure. nifiti files either have .nii extension or (older legacy format, two files per images *.hdr and *.img )
From its headers, assuming it returns foo you can get
1. the image matrix size by calling foo.header.get_data_shape()
2. The pixel size by doing foo.header.get_zooms()
3. The storage data type foo.header.get_data_dtype()
4. Then the image matrix gets returned by foo.get_fdata(), which accepts a data e.g. type foo.get_fdata(dtype=np.float32)

Load one nifti file and display some slices.

Also Display the colorbar: what intensity values are you expecting?


In [ ]:
nibTest = nib.load('C:/data_science/Dementia_PET/AD/swSPIM72_A_MBqcc_f.hdr')

In [ ]:
nibTest.header.get_zooms()

(2.0, 2.0, 2.0)

## Task 1.1: Read all the files

Use glob to make a list of all the files for all the categories.

All the images have been spatially normalized, so they're supposed to have the same matrix shape and the same pixel size. Check this.

Read all the images and store them in numpy vectors

Read also the mask image. (masks/explicit_mask.nii)

Superimpose one (randomly chosen) image with the mask contour on 3 planes.

In [ ]:
baseDir = 'D://dati_corso/Dementia_PET'
adPETL = glob.glob(os.path.join(baseDir,'AD','*.hdr'))
adPETL += glob.glob(os.path.join(baseDir,'AD','*.nii'))

bvPETL = glob.glob(os.path.join(baseDir,'bvFTD','*.hdr'))
bvPETL += glob.glob(os.path.join(baseDir,'bvFTD','*.nii'))

DLBPETL = glob.glob(os.path.join(baseDir,'DLB','*.hdr'))
DLBPETL += glob.glob(os.path.join(baseDir,'DLB','*.nii'))

hcPETL = glob.glob(os.path.join(baseDir,'HC','*.hdr'))

In [ ]:
immDim = nib.load(adPETL[0]).get_fdata().shape
hcImmVect = np.zeros(immDim+(len(hcPETL),),dtype=np.float32)
adImmVect = np.zeros(immDim+(len(adPETL),),dtype=np.float32)
bvImmVect = np.zeros(immDim+(len(bvPETL),),dtype=np.float32)
dlbImmVect = np.zeros(immDim+(len(DLBPETL),),dtype=np.float32)

In [ ]:
maskHead = nib.load(os.path.join(baseDir,'masks','explicit_mask.nii'))
maskImm = maskHead.get_fdata(dtype=np.float32)>0.5

In [ ]:
for pIdx,adSub in enumerate(adPETL):
    adImmVect[:,:,:,pIdx] = nib.load(adSub).get_fdata(dtype=np.float32)
for pIdx,hcSub in enumerate(hcPETL):
    hcImmVect[:,:,:,pIdx] = nib.load(hcSub).get_fdata(dtype=np.float32)
for pIdx,bvSub in enumerate(bvPETL):
    bvImmVect[:,:,:,pIdx] = nib.load(bvSub).get_fdata(dtype=np.float32)
for pIdx,dlbSub in enumerate(DLBPETL):
    dlbImmVect[:,:,:,pIdx] = nib.load(dlbSub).get_fdata(dtype=np.float32)

# Task 2: prepare for analysis
### Normalize image intensities

PET images are in units of tracer concentration which do not have a direct biological meaning

To be able to estimate them, we need to scale the activity to some reference value. We can do this in multiple ways: either using a reference region (known from biological knowledge not to be affected by a pathology of interest) or the "global mean".


# Task 3: Do a t-test of the images

Compare the images pixel by pixel for every pathology compared to the HC group


## Compute the "pathology patterns"

We define the average pattern of a pathology the difference between the average healthy control image and the average pathological image

In the following blocks we will show the mean patterns and save the files to disk

### Save to file

In [ ]:
headEx = nib.load(adSub)
adPatternNifti = nib.Nifti1Image(adPattern, headEx.affine, headEx.header)
nib.save(adPatternNifti,os.path.join(baseDir,'patterns','AD_pattern.nii'))
dlbPatternNifti = nib.Nifti1Image(bvPattern, headEx.affine, headEx.header)
nib.save(dlbPatternNifti,os.path.join(baseDir,'patterns','DLB_pattern.nii'))
bvFTDPatternNifti = nib.Nifti1Image(dlbPattern, headEx.affine, headEx.header)
nib.save(bvFTDPatternNifti,os.path.join(baseDir,'patterns','bvFTD_pattern.nii'))

# Task 3: Compute the "pattern expression"

We define the expression of a pattern as the projection (in the vector algebra meaning) of a patient image on the reference pattern

### Show the histograms of the scores

### Compute the effect size of the scores for the comparison of each pathology against healthy controls

### Show the scatter plot of each score against each other

Some scores are correlated

# Task 4: Implement a classifier to perform the actual classification from these scores

# Task 5: Try other classification strategies.
E.g.: try applying a PCA to the original images (After intensity scaling)
Or... PCA on log-transformed images without any scaling?? (can this be done?)

